# E4 — Calibration LM et illustration du mal-posé
**Chapitre 6 — Expériences numériques et calibration**

Cet notebook illustre :
- La calibration par Levenberg-Marquardt sans régularisation (α = 0)
- La structure de la fonction objectif L(λ, δ) : vallées plates et quasi-dégénérescence
- La sensibilité aux conditions initiales et au bruit de mesure

*Paramètres de référence : S₀=100, r=5%, σ=20%, μ_J=−10%, λ=1, δ=15%.*

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
os.makedirs('../figures', exist_ok=True)

from merton_calib.calibration import (
    synthetic_smile, compute_residuals, calibrate_tikhonov
)

plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 120})

# ── Paramètres de référence ──────────────────────────────────────────────
S0, r         = 100.0, 0.05
SIGMA0, MUJ0  = 0.20, -0.10      # fixés lors de la calibration
LAM_TRUE      = 1.0              # λ vrai
DELTA_TRUE    = 0.15             # δ vrai

STRIKES    = np.tile([80., 90., 100., 110., 120.], 4)
MATURITIES = np.repeat([0.25, 0.5, 1.0, 2.0], 5)
N          = len(STRIKES)
print(f'Grille : {N} options')

## 4.1  Surface synthétique et smile de référence

In [ ]:
sigma_mkt = synthetic_smile(S0, r, SIGMA0, MUJ0, LAM_TRUE, DELTA_TRUE,
                             STRIKES, MATURITIES, noise_std=0.0)

T_vals = np.unique(MATURITIES)
K_vals = np.unique(STRIKES)

fig, ax = plt.subplots(figsize=(7, 4))
markers = ['o','s','^','D']
colors  = ['#1f77b4','#ff7f0e','#2ca02c','#d62728']
for j, T in enumerate(T_vals):
    mask = MATURITIES == T
    ax.plot(STRIKES[mask], sigma_mkt[mask]*100, f'{markers[j]}-',
            color=colors[j], label=f'T={T}a')
ax.axhline(SIGMA0*100, ls='--', color='grey', lw=1.2, label=r'$\sigma=20\%$')
ax.set_xlabel('Strike K'); ax.set_ylabel(r'$\sigma_{\rm imp}$ (%)')
ax.set_title(r'Surface synthétique ($\lambda=1$, $\delta=15\%$)')
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('../figures/E4_surface_synthetique.pdf', bbox_inches='tight')
plt.show()

## 4.2  Carte de la fonction objectif L(λ, δ)

In [ ]:
# Évaluation sur grille 60×60
lam_g   = np.linspace(0.10, 3.5, 60)
delta_g = np.linspace(0.03, 0.45, 60)
LAM_G, DEL_G = np.meshgrid(lam_g, delta_g)

L_grid = np.empty_like(LAM_G)
for i in range(LAM_G.shape[0]):
    for j in range(LAM_G.shape[1]):
        F = compute_residuals(LAM_G[i,j], DEL_G[i,j],
                              sigma_mkt, STRIKES, MATURITIES,
                              S0, r, SIGMA0, MUJ0)
        L_grid[i,j] = np.dot(F,F) / N

idx_min = L_grid.argmin()
print(f'L_min = {L_grid.min():.2e}  à λ={LAM_G.flat[idx_min]:.3f}, δ={DEL_G.flat[idx_min]:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── Contourf log(L) ──────────────────────────────────────────────────────
ax = axes[0]
log_L  = np.log10(np.clip(L_grid, 1e-12, None))
levels = np.linspace(log_L.min(), log_L.min()+5, 30)
cp = ax.contourf(LAM_G, DEL_G, log_L, levels=levels, cmap='RdYlGn_r')
ax.contour(LAM_G, DEL_G, log_L, levels=levels[::4], colors='k', linewidths=0.4)
plt.colorbar(cp, ax=ax, label=r'$\log_{10} L(\lambda,\delta)$')
ax.plot(LAM_TRUE, DELTA_TRUE, 'w*', ms=14, label='$\\theta^*$ vrai', zorder=5,
        markeredgecolor='k', markeredgewidth=0.5)
ax.set_xlabel(r'$\lambda$'); ax.set_ylabel(r'$\delta$')
ax.set_title(r'$L(\lambda,\delta)$ — données sans bruit')
ax.legend()

# ── Profil L(λ) à δ fixé ──────────────────────────────────────────────────
ax = axes[1]
for dval, lw in zip([0.10, 0.15, 0.20, 0.25], [1, 2, 1, 1]):
    L_prof = []
    for lv in lam_g:
        F = compute_residuals(lv, dval, sigma_mkt, STRIKES, MATURITIES,
                              S0, r, SIGMA0, MUJ0)
        L_prof.append(np.dot(F,F)/N)
    lbl = fr'$\delta={int(dval*100)}\%$' + (' (vrai)' if abs(dval-0.15)<1e-9 else '')
    ax.semilogy(lam_g, L_prof, lw=lw, label=lbl)
ax.axvline(LAM_TRUE, ls='--', color='grey', lw=1)
ax.set_xlabel(r'$\lambda$'); ax.set_ylabel(r'$L(\lambda,\delta)$')
ax.set_title('Profils de L selon λ (δ fixé)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../figures/E4_contour_L.pdf', bbox_inches='tight')
plt.show()

**Lecture.** La carte de log L révèle une **vallée plate** dans la direction (λ↑, δ↓) :
de nombreux couples (λ, δ) donnent des valeurs de L quasi identiques.
Cette quasi-dégénérescence est la manifestation numérique du **mal-posé** (§ 5.2.3).

## 4.3  Calibration LM multi-départ (α = 0)

In [ ]:
res = calibrate_tikhonov(
    sigma_mkt, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
    alpha=0.0, lam0=1.0, delta0=0.15, n_starts=10, seed=42
)

print(f'λ* = {res["lambda_"]:.4f}  (vrai: {LAM_TRUE})')
print(f'δ* = {res["delta"]:.4f}  (vrai: {DELTA_TRUE})')
print(f'RMSE = {res["RMSE"]*100:.4e} vol-pts')
print(f'Itérations : {res["n_iter"]}, convergé : {res["converged"]}')

starts = res['all_starts']
lam_ends   = [s['lambda_'] for s in starts]
delta_ends = [s['delta']   for s in starts]
print(f'\nDispersion des {len(starts)} départs :')
print(f'  λ* ∈ [{min(lam_ends):.3f}, {max(lam_ends):.3f}]')
print(f'  δ* ∈ [{min(delta_ends):.3f}, {max(delta_ends):.3f}]')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
log_L  = np.log10(np.clip(L_grid, 1e-12, None))
levels = np.linspace(log_L.min(), log_L.min()+4, 25)
cp = ax.contourf(LAM_G, DEL_G, log_L, levels=levels, cmap='Blues_r', alpha=0.8)
plt.colorbar(cp, ax=ax, label=r'$\log_{10} L$')
for s in starts:
    hist = np.array(s['history']['theta'])
    if len(hist) > 1:
        ax.plot(hist[:,0], hist[:,1], '-', color='grey', lw=0.6, alpha=0.5)
    ax.plot(s['lambda_'], s['delta'], 'rs', ms=6, zorder=4)
ax.plot(LAM_TRUE, DELTA_TRUE, 'w*', ms=16, label=r'$\theta^*$ vrai', zorder=6,
        markeredgecolor='k', markeredgewidth=0.5)
ax.set_xlabel(r'$\lambda$'); ax.set_ylabel(r'$\delta$')
ax.set_title('Multi-départ LM (α = 0) — solutions finales en rouge')
ax.legend(); ax.set_xlim(0.05, 3.5); ax.set_ylim(0.02, 0.45)
plt.tight_layout()
plt.savefig('../figures/E4_multistart.pdf', bbox_inches='tight')
plt.show()

## 4.4  Instabilité — sensibilité au bruit de mesure

In [ ]:
noise_levels = [0.0, 0.001, 0.003, 0.005, 0.010]  # en vol-pts
n_trials     = 20

lam_res   = {n: [] for n in noise_levels}
delta_res = {n: [] for n in noise_levels}

rng = np.random.default_rng(0)
for noise in noise_levels:
    for _ in range(n_trials):
        sm = synthetic_smile(S0, r, SIGMA0, MUJ0, LAM_TRUE, DELTA_TRUE,
                              STRIKES, MATURITIES,
                              noise_std=noise, seed=int(rng.integers(1e6)))
        r_ = calibrate_tikhonov(sm, STRIKES, MATURITIES, S0, r, SIGMA0, MUJ0,
                                 alpha=0.0, lam0=1.0, delta0=0.15,
                                 n_starts=5, seed=42)
        lam_res[noise].append(r_['lambda_'])
        delta_res[noise].append(r_['delta'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
noise_pct  = [n*100 for n in noise_levels]
lam_stds   = [np.std(lam_res[n])   for n in noise_levels]
delta_stds = [np.std(delta_res[n]) for n in noise_levels]

axes[0].semilogy(noise_pct, lam_stds,   'bo-', ms=7, label=r'$\sigma(\lambda^*)$')
axes[0].semilogy(noise_pct, delta_stds, 'rs-', ms=7, label=r'$\sigma(\delta^*)$')
axes[0].set_xlabel('Bruit (vol-pts %)'); axes[0].set_ylabel('Écart-type du paramètre')
axes[0].set_title('Instabilité sans régularisation'); axes[0].legend()

axes[1].boxplot([lam_res[n] for n in noise_levels],
                labels=[f'{n*100:.1f}' for n in noise_levels],
                patch_artist=True,
                boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1].axhline(LAM_TRUE, ls='--', color='r', label=r'$\lambda$ vrai')
axes[1].set_xlabel('Bruit (vol-pts %)'); axes[1].set_ylabel(r'$\lambda^*$')
axes[1].set_title(r'Distribution de $\lambda^*$ selon le bruit')
axes[1].legend()
plt.tight_layout()
plt.savefig('../figures/E4_sensibilite_bruit.pdf', bbox_inches='tight')
plt.show()

print('Écart-type de λ* :')
for n, s in zip(noise_levels, lam_stds):
    print(f'  bruit={n*100:.1f}%  →  std(λ*)={s:.4f}')

**Conclusion E4.** Sur données sans bruit (α=0), LM retrouve exactement les vrais paramètres.
Avec bruit, l'écart-type de λ* croît rapidement (instabilité). La régularisation
(E5 et E6) restaure la stabilité en pénalisant l'éloignement du prior θ₀.